In [ ]:
import sys
sys.path.append("..")   # add main_folder to path

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import geopandas as gpd
import itertools
import pyarrow.dataset as pds
import pyarrow.parquet as pq
from tqdm import tqdm

from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import matplotlib.pyplot as plt


from joblib import Parallel, delayed
import pyarrow.compute as pc
from shapely.prepared import prep


import tomllib
from loguru import logger


from typing import Iterable

from src.track_generation.track_generation import simulate_tc_tracks



In [ ]:

def make_genesis_probability_map(
    df: pd.DataFrame,
    *,
    grid_deg: float = 2.0,              # grid size in degrees (e.g., 1.0 or 2.0)
    lat_range: tuple[float, float] = (-60, 60),
    lon_range: tuple[float, float] = (-180, 180),
    basin: str | None = None,           # e.g. "ATL", "EP", "WP", "NI", "SI", "SP"
    years: tuple[int, int] | None = None,
    months: set[int] | list[int] | None = None,
    title: str | None = "Cyclone Genesis Probability",
):
    """
    Plot probability of cyclone genesis on a lat-lon grid from point data.
    Expects df with columns: ['lat','lon', 'basin', 'year', 'month', ...].
    Each row is one genesis point (one genesis per SID/seed/step).

    Returns (fig, ax).
    """
    # -------- filter data (optional) --------
    data = df.copy()
    if basin is not None and "basin" in data.columns:
        data = data[data["basin"] == basin]
    if years is not None and {"year"}.issubset(data.columns):
        y0, y1 = years
        data = data[(data["year"] >= y0) & (data["year"] <= y1)]
    if months is not None and {"month"}.issubset(data.columns):
        months = set(months)
        data = data[data["month"].isin(months)]

    # Guard
    if data.empty:
        raise ValueError("No genesis points after filtering; nothing to plot.")

    # Ensure numeric lat/lon and drop NaNs
    data = data[pd.notna(data["lat"]) & pd.notna(data["lon"])].copy()
    lat = data["lat"].astype(float).to_numpy()
    lon = data["lon"].astype(float).to_numpy()

    # Wrap longitudes to [-180, 180]
    lon = ((lon + 180) % 360) - 180

    # -------- build grid & histogram --------
    lat_min, lat_max = lat_range
    lon_min, lon_max = lon_range
    lat_edges = np.arange(lat_min, lat_max + grid_deg, grid_deg, dtype=float)
    lon_edges = np.arange(lon_min, lon_max + grid_deg, grid_deg, dtype=float)

    # counts per cell
    H, lon_edges_out, lat_edges_out = np.histogram2d(
        lon, lat, bins=[lon_edges, lat_edges]
    )
    # Convert counts to probability (per cell)
    total = H.sum()
    if total <= 0:
        raise ValueError("Histogram is empty; check filters / ranges.")
    P = H / total  # probability mass per cell

    # For pcolormesh, build cell centers (optional; edges are fine)
    lon_centers = 0.5 * (lon_edges_out[:-1] + lon_edges_out[1:])
    lat_centers = 0.5 * (lat_edges_out[:-1] + lat_edges_out[1:])

    # -------- plot --------
    # Try Cartopy for coastlines; fall back to plain Matplotlib
    try:
        import cartopy.crs as ccrs
        import cartopy.feature as cfeature
        proj = ccrs.PlateCarree()
        fig = plt.figure(figsize=(10, 5))
        ax = plt.axes(projection=proj)
        # pcolormesh expects edges in PlateCarree
        mesh = ax.pcolormesh(lon_edges_out, lat_edges_out, P.T, transform=proj)
        ax.coastlines(linewidth=0.8)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4)
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=proj)
        cb = plt.colorbar(mesh, ax=ax, shrink=0.8)
        cb.set_label("Genesis probability per grid cell")
        ax.set_title(title or "Cyclone Genesis Probability")
        # gridlines (optional)
        gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
        gl.right_labels = False
        gl.top_labels = False
    except Exception:
        # Plain Matplotlib fallback (no map projection)
        fig, ax = plt.subplots(figsize=(10, 5))
        extent = [lon_edges_out.min(), lon_edges_out.max(), lat_edges_out.min(), lat_edges_out.max()]
        im = ax.imshow(
            P.T,
            origin="lower",
            extent=extent,
            aspect="auto",
        )
        cb = plt.colorbar(im, ax=ax, shrink=0.8)
        cb.set_label("Genesis probability per grid cell")
        ax.set_xlim(lon_min, lon_max)
        ax.set_ylim(lat_min, lat_max)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.set_title(title or "Cyclone Genesis Probability")
        ax.grid(alpha=0.3)

    return fig, ax

In [ ]:

def _wrap_to_180(a):
    # Map longitudes to [-180, 180)
    return ((np.asarray(a, dtype=float) + 180.0) % 360.0) - 180.0

def _split_dateline_segments(lons, lats):
    """Return list of (lon_seg, lat_seg) arrays, split where |Δlon| > 180°."""
    lons = np.asarray(lons, dtype=float)
    lats = np.asarray(lats, dtype=float)

    # Ensure plotting-friendly longitudes
    lons = _wrap_to_180(lons)

    # Find big jumps in longitude between consecutive points
    jumps = np.abs(np.diff(lons))
    breaks = np.where(jumps > 180.0)[0] + 1

    lon_segs = np.split(lons, breaks)
    lat_segs = np.split(lats, breaks)
    return list(zip(lon_segs, lat_segs))

def plot_cyclone_tracks_map(df, sid_col="SID", lat_col="lat", lon_col="lon", time_col=None,
                            show_legend=False, with_markers=False):
    fig, ax = plt.subplots(figsize=(12, 6),
                           subplot_kw={"projection": ccrs.PlateCarree()})
    ax.set_global()
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.gridlines(draw_labels=True)

    # One color per SID is optional; avoid clutter if many storms
    for sid, group in df.groupby(sid_col, sort=False):
        g = group
        if time_col is not None and time_col in g.columns:
            g = g.sort_values(time_col)

        lon_vals = g[lon_col].to_numpy()
        lat_vals = g[lat_col].to_numpy()

        for x, y in _split_dateline_segments(lon_vals, lat_vals):
            if len(x) < 2:
                continue
            ax.plot(x, y,
                    transform=ccrs.PlateCarree(),
                    linewidth=1.4, alpha=0.9)
            if with_markers:
                ax.plot(x, y,
                        transform=ccrs.PlateCarree(),
                        marker="o", markersize=2, linestyle="None", alpha=0.9)

        if show_legend:
            ax.plot([], [], label=str(sid))  # dummy handle

    ax.set_title("Cyclone Tracks (dateline-safe)")
    if show_legend:
        ax.legend(title=sid_col, fontsize=8, loc="upper right", bbox_to_anchor=(1.2, 1))
    plt.tight_layout()
    plt.show()

In [ ]:
config_path = "../config.toml"
with open(
    config_path,
    "rb",
) as f:  # Open the file in binary mode
    config_files = tomllib.load(f)

In [ ]:
main_config = config_files["main_params"]
gen_config = config_files["generation"]
intens_config = config_files["intensification"]
n_seeds = gen_config["n_seeds"]

catherina_fit_path = '..' / Path(main_config["data_dir"]) / "fit"
cyclones_dir = Path(gen_config["synthetic_tracks_dir"])

ibtracks_file_path = '..' / Path(gen_config["ibtracs_path"])

data_dir = ".." / Path(main_config["data_dir"])

ne_10m_coastline_zip = ".." / Path(main_config["data_dir"]) / "ne_10m_coastline.zip"
ne_10m_land_zip = ".." /Path(main_config["data_dir"]) / "ne_10m_land.zip"
model_exp_pbar = tqdm(
    list(itertools.product(main_config["models"], main_config["experiments"]))
)
land = gpd.read_file(ne_10m_land_zip).union_all()
land_union = prep(land)


In [ ]:
genesis_ds = pds.dataset(
    '..' / Path(main_config["data_dir"]) / "catherina_ssp585" / "genesis",
    format="parquet",
    partitioning="hive",
)
df = genesis_ds.to_table().to_pandas()


In [ ]:
make_genesis_probability_map(df.loc[lambda row:(row['seed']==1)])

In [ ]:
from tqdm_joblib import tqdm_joblib
import more_itertools


genesis_ds = pds.dataset(
    '..' / Path(main_config["data_dir"]) / "catherina_ssp585" / "genesis",
    format="parquet",
    partitioning="hive",
)

batch_seeds_size = 1

batch_seeds = list(more_itertools.chunked(list(range(n_seeds)), n=batch_seeds_size))

# Remove default handler (console)
logger.remove()
logfile = "debug.log"
# Add a file sink for debugging logs
logger.add(
    logfile, rotation="10 MB", enqueue=True
)  # Rotates file when it reaches 10MB

save_tracks = '..' / Path(main_config["data_dir"])/"catherina_ssp585"

with tqdm_joblib(batch_seeds, desc="Processing seeds", position=0, total=len(batch_seeds), leave=True,
) as progress_bar:
    Parallel(n_jobs=4, backend="loky")(
        delayed(simulate_tc_tracks)(
            genesis_ds= genesis_ds,
            catherina_fit_path = catherina_fit_path,
            max_steps=175,
            seeds=seeds,
            logfile=logfile,
            save_dir=save_tracks,
        )
        for batch_id, seeds in enumerate(batch_seeds)
    )


In [ ]:
track_generation_ds = pds.dataset(
    '..' / Path(main_config["data_dir"]) / "catherina_test6_to_delete" / "tracks",
    format="parquet",
    partitioning="hive",
)
df = track_generation_ds.to_table().to_pandas()

In [ ]:
plot_cyclone_tracks_map(df)